# Enriquecer una tabla de transparencia con datos extra?dos por OCR

Este ejemplo parte de `transparencia_telecom_dic_2024.csv`, descarga un grupo peque?o de resoluciones enlazadas en el CSV, reconoce su texto con **Tesseract** y agrega a cada fila las columnas `Conclusi?n OCR`, `Resultado OCR`, `Entidad recurrida OCR` y `Estado OCR`.

La asociaci?n se hace por la fila y el enlace de origen. El OCR y las reglas de texto son aproximados: revisa cada campo contra el PDF antes de usarlo. El ejemplo procesa solo unas pocas filas para que sea manejable en clase.

## Preparaci?n en Windows

Ejecuta la celda siguiente para instalar los paquetes de Python requeridos (`pandas`, `requests`, `pdf2image`, `pytesseract` y `Pillow`). Tesseract con el idioma espa?ol (`spa`) y Poppler son programas externos y deben instalarse aparte. Si no est?n en `PATH`, indica sus rutas en la celda de configuraci?n. El CSV de pr?ctica se busca junto al cuaderno o en `sesiones/sesion-03`. La descarga de resoluciones requiere conexi?n a internet.

### Instalar Tesseract con winget

En PowerShell, instala Tesseract desde el repositorio de winget. Durante la instalación, incluye los datos de idioma Spanish (`spa`).

```powershell
winget install --id UB-Mannheim.TesseractOCR --exact --source winget
```

Abre una terminal nueva y verifica que Tesseract esté disponible y que aparezca `spa` entre los idiomas instalados:

```powershell
tesseract --version
tesseract --list-langs
```

Si el comando no se reconoce, agrega la carpeta de instalación de Tesseract al `PATH` o configura `TESSERACT_CMD` en la celda de configuración.

In [ ]:
%pip install pandas requests pdf2image pytesseract Pillow

In [ ]:
from pathlib import Path

roots = [Path.cwd(), *Path.cwd().parents]
candidates = [root / 'transparencia_telecom_dic_2024.csv' for root in roots]
candidates += [root / 'sesiones' / 'sesion-03' / 'transparencia_telecom_dic_2024.csv' for root in roots]
CSV_PATH = next((path for path in candidates if path.is_file()), candidates[0])
WORK_DIR = CSV_PATH.parent / 'ejemplo_ocr_transparencia'
PDF_DIR = WORK_DIR / 'pdf'
PDF_DIR.mkdir(parents=True, exist_ok=True)
MAX_DOCUMENTS = 3  # Aumenta gradualmente despu?s de revisar el resultado.
REQUEST_DELAY_SECONDS = 1.0
DPI = 250
TESSERACT_CMD = None  # Ej.: r'C:\Program Files\Tesseract-OCR\tesseract.exe'
POPPLER_PATH = None   # Ej.: r'C:\poppler\Library\bin'

if not CSV_PATH.is_file():
    raise FileNotFoundError(f'No se encontr? el CSV junto al cuaderno: {CSV_PATH}')
print(f'CSV: {CSV_PATH}')
print(f'PDF y resultados: {WORK_DIR}')

## 1. Cargar y preparar las filas del CSV

El CSV usa `;` como separador. Se intenta leer en UTF-8 y luego en `latin-1`. Extraemos la URL del enlace HTML y usamos la fila original como identificador para incorporar despu?s los campos OCR sin perder sus datos de transparencia.

In [ ]:
import re
import pandas as pd

try:
    df = pd.read_csv(CSV_PATH, sep=';', encoding='utf-8-sig', dtype=str)
except UnicodeDecodeError:
    df = pd.read_csv(CSV_PATH, sep=';', encoding='latin-1', dtype=str)

df = df.loc[:, [(str(c).strip() != '') and not str(c).startswith('Unnamed') for c in df.columns]].copy()
LINK_COLUMN = 'Vinculo al texto integro y actualizado'
if LINK_COLUMN not in df.columns:
    raise KeyError(f'No se encontr? la columna {LINK_COLUMN!r}. Columnas: {list(df.columns)}')

def extract_url(value):
    if not isinstance(value, str):
        return None
    match = re.search(r'''href\s*=\s*['\"]([^'\"]+)['\"]''', value, flags=re.IGNORECASE)
    return match.group(1) if match else None

YEAR_COLUMN = df.columns[0]  # Primera columna: a?o.
NUMBER_COLUMN = next((c for c in df.columns if 'mero del acto' in str(c)), None)
if NUMBER_COLUMN is None:
    raise KeyError('No se encontr? la columna de n?mero del acto.')
df['_fila_origen'] = range(len(df))
df['_url_documento'] = df[LINK_COLUMN].map(extract_url)
con_url = df[df['_url_documento'].notna()].head(MAX_DOCUMENTS).copy()
print(f'Filas en el CSV: {len(df):,}')
print(f'Documentos seleccionados para esta demostraci?n: {len(con_url)}')
display(con_url[['_fila_origen', YEAR_COLUMN, 'Mes', NUMBER_COLUMN, '_url_documento']].head())

## 2. Descargar una muestra de resoluciones

Cada descarga se valida como PDF antes de guardarla. Conservamos la fila de origen junto al archivo para que las columnas extra?das vuelvan al registro correcto. Para reducir solicitudes, la muestra est? limitada por `MAX_DOCUMENTS`.

In [ ]:
import time
import requests

registros = []
for _, row in con_url.iterrows():
    numero_fila = int(row['_fila_origen'])
    url = row['_url_documento']
    destino = PDF_DIR / f'resolucion_fila_{numero_fila}.pdf'
    registro = {'_fila_origen': numero_fila, '_url_documento': url,
                '_pdf_local': str(destino), 'estado_descarga': 'error', 'error_descarga': ''}
    try:
        response = requests.get(url, timeout=40, headers={'User-Agent': 'Mozilla/5.0'})
        response.raise_for_status()
        if not response.content.startswith(b'%PDF'):
            raise ValueError('El enlace no devolvi? un archivo PDF.')
        destino.write_bytes(response.content)
        registro['estado_descarga'] = 'ok'
        print(f'Fila {numero_fila}: PDF descargado ({len(response.content):,} bytes)')
    except Exception as exc:
        registro['error_descarga'] = f'{type(exc).__name__}: {exc}'
        print(f'Fila {numero_fila}: no se pudo descargar: {registro["error_descarga"]}')
    registros.append(registro)
    time.sleep(REQUEST_DELAY_SECONDS)

descargas = pd.DataFrame(registros)
display(descargas[[' _fila_origen'.strip(), 'estado_descarga', 'error_descarga']])

## 3. Reconocer las p?ginas con Tesseract

Renderizamos cada PDF a im?genes y usamos exclusivamente Tesseract con el idioma espa?ol. Guardamos el texto completo para poder inspeccionar qu? ley? el motor antes de extraer campos.

In [ ]:
from pdf2image import convert_from_path
import pytesseract

if TESSERACT_CMD:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

try:
    idiomas = pytesseract.get_languages(config='')
except Exception as exc:
    raise RuntimeError('No se pudo ejecutar Tesseract. Revisa la instalaci?n y TESSERACT_CMD.') from exc
if 'spa' not in idiomas:
    raise RuntimeError(f'Falta el idioma spa en Tesseract. Idiomas instalados: {idiomas}')

ocr_rows = []
for item in descargas.to_dict('records'):
    resultado = {**item, 'texto_ocr': '', 'estado_ocr': 'omitido', 'error_ocr': ''}
    if item['estado_descarga'] != 'ok':
        resultado['error_ocr'] = 'No hay PDF v?lido para procesar.'
        ocr_rows.append(resultado)
        continue
    try:
        pages = convert_from_path(item['_pdf_local'], dpi=DPI, poppler_path=POPPLER_PATH)
        textos = [pytesseract.image_to_string(page, lang='spa', config='--psm 3') for page in pages]
        resultado['texto_ocr'] = '\n'.join(f'--- P?GINA {pagina} ---\n{texto}' for pagina, texto in enumerate(textos, start=1)).strip()
        resultado['estado_ocr'] = 'ok' if resultado['texto_ocr'] else 'sin_texto'
        print(f"Fila {item['_fila_origen']}: {len(pages)} p?gina(s), {len(resultado['texto_ocr']):,} caracteres OCR")
    except Exception as exc:
        resultado['estado_ocr'] = 'error'
        resultado['error_ocr'] = f'{type(exc).__name__}: {exc}'
        print(f"Fila {item['_fila_origen']}: error OCR: {resultado['error_ocr']}")
    ocr_rows.append(resultado)

ocr = pd.DataFrame(ocr_rows)
display(ocr[['_fila_origen', 'estado_ocr', 'error_ocr']])

## 4. Convertir el texto OCR en campos y agregarlos a la tabla

Como ejercicio, buscamos el p?rrafo que sigue a `RESUELVE` y aplicamos reglas sencillas para proponer el resultado y la entidad. Son heur?sticas de demostraci?n: un OCR puede confundir letras y una regla puede no cubrir todas las formas de redacci?n.

In [ ]:
def extract_conclusion(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    # Toma el texto despu?s de RESUELVE hasta el pr?ximo encabezado de p?gina o fin.
    match = re.search(r'RESUELVE\s*[:;]?\s*(.*?)(?=\n\s*\n|\n\s*---\s*P[A?]GINA|\Z)', text,
                      flags=re.IGNORECASE | re.DOTALL)
    if not match:
        return ''
    return re.sub(r'\s+', ' ', match.group(1)).strip()

def classify_result(conclusion):
    upper = conclusion.upper()
    if 'ACOGER PARCIALMENTE' in upper:
        return 'ACOGE PARCIALMENTE'
    if 'ACOGER' in upper:
        return 'ACOGE'
    if any(term in upper for term in ('RECHAZAR', 'DESESTIMAR', 'NO HA LUGAR')):
        return 'DESESTIMA'
    return 'INDETERMINADO' if not conclusion else 'REVISAR'

def extract_entity(conclusion):
    if not conclusion:
        return ''
    match = re.search(r'contra\s+de\s+(.+?)(?:,|(?=\.\s*$)|$)', conclusion,
                      flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', match.group(1)).strip() if match else ''

ocr['Conclusi?n OCR'] = ocr['texto_ocr'].map(extract_conclusion)
ocr['Resultado OCR'] = ocr['Conclusi?n OCR'].map(classify_result)
ocr['Entidad recurrida OCR'] = ocr['Conclusi?n OCR'].map(extract_entity)

# Adjunta los campos nuevos a las filas originales, conservando todas las columnas del CSV.
campos = ocr[['_fila_origen', '_url_documento', 'estado_ocr', 'Conclusi?n OCR',
              'Resultado OCR', 'Entidad recurrida OCR', 'error_ocr']]
tabla_enriquecida = df.merge(campos, on='_fila_origen', how='left', suffixes=('', '_ocr'))
tabla_enriquecida['Estado OCR'] = tabla_enriquecida['estado_ocr'].fillna('no procesado')
display(tabla_enriquecida.loc[tabla_enriquecida['_fila_origen'].isin(con_url['_fila_origen']),
    [NUMBER_COLUMN, 'Fecha', 'Conclusi?n OCR', 'Resultado OCR', 'Entidad recurrida OCR', 'Estado OCR']])

## 5. Revisar y guardar el resultado

La nueva tabla conserva las columnas del CSV y a?ade los datos propuestos por OCR. Revisa primero `texto_ocr` y compara los campos nuevos con el documento original. Los registros que no se descargaron o no pudieron leerse quedan identificados en `Estado OCR` y `error_ocr`.

In [ ]:
# Muestra para inspecci?n manual
for _, item in ocr.iterrows():
    print(f"\n--- Fila CSV {item['_fila_origen']} ? {item['estado_ocr']} ---")
    print(item['texto_ocr'][:2500] if item['texto_ocr'] else item['error_ocr'])

OUTPUT_CSV = WORK_DIR / 'transparencia_telecom_con_ocr.csv'
export = tabla_enriquecida.drop(columns=['_fila_origen', '_url_documento', 'estado_ocr', 'error_ocr'], errors='ignore')
export.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig', sep=';')
print(f'\nTabla enriquecida guardada en: {OUTPUT_CSV}')


### Qu? demuestra este ejemplo

1. El CSV aporta metadatos existentes, como n?mero y fecha del acto.
2. El enlace de cada fila permite descargar el documento correspondiente.
3. Tesseract aporta texto que no estaba disponible como columna en la tabla.
4. Reglas transparentes convierten una parte del texto en campos nuevos.
5. La asociaci?n por fila de origen agrega esos campos a la tabla sin perder su v?nculo con la resoluci?n.

La extracci?n autom?tica propone datos y facilita el an?lisis a escala; la revisi?n del PDF sigue siendo necesaria, especialmente para nombres, n?meros y conclusiones.